In [ ]:
# ----------------------------
# CNN PINN simulator
# ----------------------------

# Import necessary libraries
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import numpy as np
from scipy.ndimage import distance_transform_edt
import os
import time
import pandas as pd

import os
import json
from pathlib import Path
import pyvista as pv


In [ ]:
# ----------------------------
# CNN PINN Block Definitions
# ----------------------------

class ConvBlock(nn.Module):
        """Two 3x3 convs, each followed by GroupNorm + SiLU."""
        def __init__(self, in_ch, out_ch, groups=8):
            super().__init__()
            groups = min(groups, out_ch)
            self.net = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.GroupNorm(groups, out_ch),
                nn.SiLU(),
                nn.Conv2d(out_ch, out_ch, 3, padding=1),
                nn.GroupNorm(groups, out_ch),
                nn.SiLU(),
            )
        
        def forward(self, x):
            return self.net(x)
    
class DownBlock(nn.Module):
    """Stride-2 conv downsample + ConvBlock.
    Learned (strided-conv) downsampling instead of max-pooling -- it preserves more boundary-layer detail,
    which matters for near-wall gradients that impact physics loss"""

    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.down = nn.Conv2d(in_ch, in_ch, 4, stride=2, padding=1)
        self.block = ConvBlock(in_ch, out_ch)
    
    def forward(self, x):
        return self.block(self.down(x))

class UpBlock(nn.Module):
    """Transpose-conv upsample, concat skip connection, ConvBlock."""
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 4, stride=2, padding=1)
        self.block = ConvBlock(in_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            # guards against off-by-one size mismatches on odd input dims
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.block(x)
    
# ----------------------------
# Spectral Convolution Layer for the bottleneck to solve pressure across the whole domain
# ----------------------------

class SpectralConv2d(nn.Module):
    """2D Fourier layer. This is the spectral convolution layer used in the bottleneck of the CNN PINN architecture."""
    def __init__(self, in_ch, out_ch, modes_h, modes_w):
        super().__init__()
        self.in_ch = in_ch
        self.out_ch = out_ch
        self.modes_h = modes_h
        self.modes_w = modes_w
        self.scale = (1.0 / (in_ch * out_ch))
        self.weights = nn.Parameter(self.scale * torch.rand(in_ch, out_ch, modes_h, modes_w, dtype=torch.cfloat))

    def forward(self, x):
        B, C, H, W = x.shape
        x_ft = torch.fft_rfft2(x, norm="ortho") # (B, C, H, W//2+1), complex

        out_ft = torch.zeros(B, self.out_ch, H, W // 2 + 1, dtype=torch.cfloat, device=x.device)
        mh = min(self.modes_h, H)
        mw = min(self.modes_w, W // 2 + 1)

        out_ft[:, :, :mh, :mw] = torch.einsum(
            "bixy,ioxy->boxy", x_ft[:, :, :mh, :mw], self.weights[:, :, :mh, :mw]
        )

        return torch.fft.irfft2(out_ft, s=(H, W), norm="ortho")
    
class FNOBlock(nn.Module):
    """Fourier Neural Operator block. This is the bottleneck of the CNN PINN architecture."""
    def __init__(self, channels, modes_h=16, modes_w=16):
        super().__init__()
        self.spectral = SpectralConv2d(channels, channels, modes_h, modes_w)
        self.pointwise = nn.Conv2d(channels, channels, 1)
        self.norm = nn.GroupNorm(min(8, channels), channels)
        self.act = nn.SiLU()

    def forward(self, x):
        out = self.spectral(x) + self.pointwise(x)
        return self.act(self.norm(out))
        

In [ ]:
class CNN_PINN(nn.Module):
    """
    U-Net encoder-deecoder with an FNO bottleneck.
    
    Input: (B, 4, H, W) -- occupancy, SDF, x-coord, y-coord
    Output: (B, 4, H, W) -- u, v, p, T (fluid region values only, masked in solid region)

    Sized for H=100, W = 200 (25x50 bottleneck, small enough for FFT-based spectral convs to be cheap)
    """
    def __init__(self, in_ch=4, base_ch=32, n_fno_blocks=4, fno_modes=16):
        super().__init__()

        self.stem = ConvBlock(in_ch, base_ch)               # H x   W
        self.down1 = DownBlock(base_ch, base_ch * 2)        # H/2 x W/2
        self.down2 = DownBlock(base_ch * 2, base_ch * 4)    # H/4 x W/4

        self.bottleneck = nn.Sequential(
            *[FNOBlock(base_ch * 4, fno_modes, fno_modes) for _ in range(n_fno_blocks)]
        )

        self.up2 = UpBlock(base_ch * 4, base_ch * 2, base_ch * 2)  # H/2 x W/2
        self.up1 = UpBlock(base_ch * 2, base_ch, base_ch)
        self.head = nn.Conv2d(base_ch, 4, kernel_size=1)  # u, v, p, T

    def forward(self, x):
        s0 = self.stem(x)
        s1 = self.down1(s0)
        s2 = self.down2(s1)
        b = self.bottleneck(s2)
        u2 = self.up2(b, s1)
        u1 = self.up1(u2, s0)
        out = self.head(u1)
        return out

In [ ]:
class SpatialDerivative(nn.Module):
    """
    Calculates spatial derivative operators via fixed conv2d kernels using central difference approximations.
    Replicate-padded to get a valid derivative at the boundary. This is used to compute the physics loss for the PINN.
    """
    def __init__(self, dx: float, dy: float):
        super().__init__()
        assert abs(dx - dy) < 1e-9, "assumes square pixels (dx == dy)"
        self.h = dx
        
        ddx = torch.tensor([[0, 0, 0], [-1, 0, 1], [0, 0, 0]], dtype=torch.float32) / (2 * dx)
        ddy = torch.tensor([[0, -1, 0], [0, 0, 0], [0, 1, 0]], dtype=torch.float32) / (2 * dy)
        lap = torch.tensor([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=torch.float32) / (dx * dy)

        self.register_buffer("ddx", ddx.view(1, 1, 3, 3))
        self.register_buffer("ddy", ddy.view(1, 1, 3, 3))
        self.register_buffer("lap", lap.view(1, 1, 3, 3))

    def _conv_deriv(self, f, kernel):
        f = F.pad(f, (1, 1, 1, 1), mode="replicate")
        return F.conv2d(f, kernel)

    def d_dx(self, f):
        return self._conv_deriv(f, self.ddx)
    
    def d_dy(self, f):
        return self._conv_deriv(f, self.ddy)
    
    def laplacian(self, f):
        return self._conv_deriv(f, self.lap)

In [ ]:
class PhysicsLoss(nn.Module):
    """
    Computes PDE residual + boundary-condition losses for a single fixed
    set of fluid properties / boundary conditions.

    forward() returns a dict of individual loss terms, which can be weighted and summed to form the total loss.
    """

    def __init__(self, fluid_properties: dict, boundary_conditions: dict, domain_size=(2.0, 1.0), grid_shape=(200, 100)):
        super().__init__()
        W, H = grid_shape
        dx = domain_size[0] / W
        dy = domain_size[1] / H
        self.deriv = SpatialDerivative(dx, dy)

        self.rho = fluid_properties["rho"]
        self.mu = fluid_properties["mu"]
        self.k = fluid_properties["k"]
        self.cp = fluid_properties["cp"]
        self.nu = self.mu / self.rho  # kinematic viscosity
        self.alpha = self.k / (self.rho * self.cp)  # thermal diffusivity

        self.U_in = boundary_conditions["U_in"]
        self.T_in = boundary_conditions["T_in"]
        self.T_wall = boundary_conditions["T_wall"]
        self.P_pin = boundary_conditions["P_pin"]

    def dilate(self, mask, iterations=1):
        for _ in range(iterations):
            mask = F.pad(mask, (1, 1, 1, 1), mode="replicate")
            mask = F.max_pool2d(mask, kernel_size=3, stride=1, padding=0)
        return mask

    def erode(self, mask, iterations=1):
        return 1.0 - self.dilate(1.0 - mask, iterations=iterations)
    
    def masked_mse(self, field: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        diff2 = (field - target) ** 2 * mask
        denom = mask.sum().clamp(min=1.0)
        return diff2.sum() / denom

    def forward(self, pred: torch.Tensor, occupancy: torch.Tensor) -> dict:
        """
        Args:
            pred: (B, 4, H, W) network output -- channels [u, v, p, T]
            occupancy: (B, 1, H, W) -- 1 = solid, 0 = fluid
        
        Returns:
            dict of scalar loss tensors: pde_continuity, pde_momentum_x, pde_momentum_y,
            pde_energy, bc_inlet, bc_wall, bc_outlet, bc_pressure_pin
        """
        u, v, p, T = pred[:, 0:1], pred[:, 1:2], pred[:, 2:3], pred[:, 3:4]
        fluid = 1-occupancy  # 1 = fluid, 0 = solid
        solid = occupancy

        # Masks
        wall_band = fluid * self.dilate(solid, iterations = 1)
        interior = self.erode(fluid, iterations=1)

        inlet_mask = torch.zeros_like(fluid)
        inlet_mask[:, :, :, 0] = fluid[:, :, :, 0]  # left edge
        outlet_mask = torch.zeros_like(fluid)
        outlet_mask[:, :, :, -1] = fluid[:, :, :, -1]  # right edge
        
        # PDE residuals
        du_dx, du_dy = self.deriv.d_dx(u), self.deriv.d_dy(u)
        dv_dx, dv_dy = self.deriv.d_dx(v), self.deriv.d_dy(v)
        dp_dx, dp_dy = self.deriv.d_dx(p), self.deriv.d_dy(p)
        dT_dx, dT_dy = self.deriv.d_dx(T), self.deriv.d_dy(T)

        continuity = du_dx + dv_dy
        momentum_x = (u * du_dx + v * du_dy) + (1 / self.rho) * dp_dx - self.nu * (self.deriv.laplacian(u))
        momentum_y = (u * dv_dx + v * dv_dy) + (1 / self.rho) * dp_dy - self.nu * (self.deriv.laplacian(v))
        energy = (u * dT_dx + v * dT_dy) - self.alpha * (self.deriv.laplacian(T))

        losses = {
            "pde_continuity": self.masked_mse(continuity, torch.zeros_like(continuity), interior),
            "pde_momentum_x": self.masked_mse(momentum_x, torch.zeros_like(momentum_x), interior),
            "pde_momentum_y": self.masked_mse(momentum_y, torch.zeros_like(momentum_y), interior),
            "pde_energy": self.masked_mse(energy, torch.zeros_like(energy), interior)
            }
        
        # Boundary condition losses
        inlet_loss = self.masked_mse(u, 0.0, inlet_mask) + self.masked_mse(v, 0.0, inlet_mask) + self.masked_mse(T, self.T_in, inlet_mask)
        outlet_loss = self.masked_mse(du_dx, 0.0, outlet_mask) + self.masked_mse(dv_dx, 0.0, outlet_mask) + self.masked_mse(dT_dx, 0.0, outlet_mask)

        # add wall and pressure bc losses later
        losses["bc_inlet"] = inlet_loss
        losses["bc_outlet"] = outlet_loss

        return losses

In [ ]:
class PINN_Simulator(nn.Module):
    def __init__(self, fluid_properties: dict, boundary_conditions: dict, domain_size=(2.0, 1.0), grid_shape=(200, 100)):
        super().__init__()
        self.fluid_properties = fluid_properties
        self.boundary_conditions = boundary_conditions
        self.domain_size = domain_size
        self.grid_shape = grid_shape

        self.model = CNN_PINN(in_ch=4, base_ch=32, n_fno_blocks=4, fno_modes=16)
        self.physics_loss = PhysicsLoss(fluid_properties, boundary_conditions, domain_size, grid_shape)
        self.weights = {
            "pde_continuity": 1.0,
            "pde_momentum_x": 1.0,
            "pde_momentum_y": 1.0,
            "pde_energy": 1.0,
            "bc_inlet": 1.0,
            "bc_outlet": 1.0
        }

    def build_input_tensor(occupancy_grid: np.ndarray, domain_size=(2.0,1.0)) -> torch.Tensor:
        """
        Convert a binary occupancy grid (1 = solid, 0 = fluid) into the
        multi-channel input tensor for the network.
    
        Channels:
            0: occupancy            (0 = fluid, 1 = solid)
            1: signed distance fn   (positive in fluid, negative in solid),
                                    normalized by the domain diagonal
            2: normalized x coord   in [0, 1]
            3: normalized y coord   in [0, 1]
    
        Coordinate + SDF channels matter because plain convolutions are
        translation-equivariant, but inlet/outlet/wall locations are fixed in
        the domain -- the network needs to know *where* it is, and the SDF gives
        a smooth signal near walls instead of a hard 0/1 jump.
    
        Args:
            occupancy_grid: (H, W) array, 1 = solid, 0 = fluid
            domain_size: (Lx, Ly) physical domain size in meters
    
        Returns:
            Tensor of shape (4, H, W), float32
        """
        occ = occupancy_grid.astype(np.float32)
        H, W = occ.shape
    
        dx = domain_size[0] / W
        dy = domain_size[1] / H
        assert abs(dx - dy) < 1e-9, "build_input_tensor assumes square pixels (dx == dy)"
    
        dist_to_solid = distance_transform_edt(occ == 0)  # fluid px -> nearest solid px
        dist_to_fluid = distance_transform_edt(occ == 1)  # solid px -> nearest fluid px
        sdf_px = np.where(occ == 0, dist_to_solid, -dist_to_fluid)
        sdf_m = sdf_px * dx
        diag = np.sqrt(domain_size[0] ** 2 + domain_size[1] ** 2)
        sdf_norm = (sdf_m / diag).astype(np.float32)
    
        y_coords, x_coords = np.meshgrid(
            np.linspace(0.0, 1.0, H, dtype=np.float32),
            np.linspace(0.0, 1.0, W, dtype=np.float32),
            indexing="ij",
        )
    
        input_np = np.stack([occ, sdf_norm, x_coords, y_coords], axis=0)
        return torch.from_numpy(input_np)
    
    def combine_losses(self, loss_dict: dict, weights: dict) -> torch.Tensor:
        total = 0.0
        for name, value in loss_dict.items():
            total += weights.get(name, 1.0) * value
        return total
    
    def load_dataset(self, config_dir: str, results_dir: str, domain_size: tuple = (2.0, 1.0), grid_shape: tuple = (200, 100)) -> Dataset:
        """
        Load a dataset of occupancy grids and corresponding fluid flow solutions.
        The dataset is expected to be in the form of .npz files containing:
            - occupancy: (H, W) array, 1 = fluid, 0 = solid
            - u: (H, W) array, x-velocity
            - v: (H, W) array, y-velocity
            - p: (H, W) array, pressure
            - T: (H, W) array, temperature
            Args:
                config_dir: directory containing the configuration files for the dataset
                results_dir: directory containing the results of the simulations
                domain_size: physical domain size in meters
                grid_shape: shape of the grid (W, H)

            Returns:
                Dataset: the loaded dataset formatted as a pandas DataFrame with columns for occupancy, u, v, p, T, and any other relevant metadata.
            """
        
    
    def run(self, occupancy_grid: np.ndarray, domain_size: tuple = (2.0, 1.0)):
        """
        Run the PINN simulator on a given occupancy grid.
    
        Args:
            occupancy_grid: (H, W) array, 1 = fluid, 0 = solid
            domain_size: (Lx, Ly) physical domain size in meters
    
        Returns:
            Tensor of shape (4, H, W), float32 -- predicted u, v, p, T fields
        """
        input_tensor = self.build_input_tensor(occupancy_grid, domain_size)
        input_tensor = input_tensor.unsqueeze(0)  # add batch dimension
        with torch.no_grad():
            output = self.forward(input_tensor)
        return output.squeeze(0)  # remove batch dimension

    def train_model(self, 
                    config_dir: str,
                    data_dir: str,
                    checkpoint_dir: str,
                    pinn_result_dir: str,
                    fluid_properties: dict,
                    boundary_conditions: dict,
                    domain_size: tuple = (2.0, 1.0),
                    grid_shape: tuple = (200, 100),
                    batch_size: int = 4,
                    num_epochs: int = 100,
                    warmup_epochs: int = 10,
                    ramp_epochs: int = 10,
                    learning_rate: float = 1e-4,
                    device: str = "cpu",
                    num_workers: int = 4
                    ):
        """
        Train the PINN simulator on a dataset of occupancy grids and corresponding fluid flow solutions.
        """
        device = device
        os.makedirs(checkpoint_dir, exist_ok=True)
        os.makedirs(pinn_result_dir, exist_ok=True)

        # Load dataset
        

In [ ]:
verbose = False
# extract the occupancy grid and domain size from the config file
def read_config(config_dir: str) -> dict:
    """
    Read the configuration files for the dataset and return a dictionary of parameters.
    """
    config_path = Path(config_dir) / "config.json"
    with open(config_path, "r") as f:
        config = json.load(f)
    
    return {
        "occupancy_grid": np.array(config["occupancy_grid"], dtype=np.float32),
        "threshold": float(config["threshold"]),
        "grid_nx": int(config["grid_nx"]),
        "grid_ny": int(config["grid_ny"]),
        "domain_length": float(config["domain_length"]),
        "domain_height": float(config["domain_height"]),
    }

# Names of fileds as stored in the Exodus file
EXODUS_VEL_NAME = "vel_"        # 3-component vector field (u, v, w)
EXODUS_PRES_NAME = "p"          # scalar pressure field
EXODUS_TEMP_NAME = "T"          # scalar temperature field

# extract and interpolate the results from the MOOSE simulation output .e files
def load_exodus_mesh(exodus_path: str | Path) -> pv.DataSet:
    """
    Read an Exodus II file and return a single merged PyVista dataset
    at the final time step, with cell data promoted to point data.
    
    Parameters
    ----------
    exodus_path : str | Path
        Path to the Exodus II file.
    
    Returns
    -------
    pv.DataSet
        Merged PyVista dataset.
    """
    exodus_path = str(exodus_path)
    reader = pv.get_reader(exodus_path)
    reader.set_active_time_value(reader.time_values[-1])  # last time step
    mb = reader.read()

    # Filter out empty blocks before mergine - avoids PyVista warnings
    valid_blocks = _extract_valid_blocks(mb)
    if not valid_blocks:
        raise RuntimeError(
            f"No non-empty blocks found in {exodus_path}. "
            "The simulation may have failed or written an empty results file."
        )
    
    mesh = pv.MultiBlock(valid_blocks).combine()
    mesh = mesh.cell_data_to_point_data()
    return mesh

def _extract_valid_blocks(block) -> list:
    """
    Recursively collect all non-empty leaf datasets from a PyVista MultiBlock tree.
    A block is 'valid' if it has at least one point and one cell.
    """
    datasets = []
    if isinstance(block, pv.MultiBlock):
        for i in range(block.n_blocks):
            datasets.extend(_extract_valid_blocks(block[i]))
    else:
        if block is None:
            return []
        if block.n_points == 0 or block.n_cells == 0:
            return []
        datasets.append(block)
    return datasets

def extract_fields(
        mesh: pv.DataSet,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Extract raw node coordinates and field arrays from a merged
    PyVista mesh produced by load_exodus_mesh().

    The velocity field is stored in the Exodus file as a 3-component
    vectore named 'vel_' (MOOSE INS convention). Components are split
    into u (x-velocity) and v (y-velocity).

    Returns
    -------
    coords : ndarray, shape (N, 3) - x, y, z node coordinates
    u      : ndarray, shape (N,)   - x-velocity at each node
    v      : ndarray, shape (N,)   - y-velocity at each node
    p      : ndarray, shape (N,)   - pressure at each node
    T      : ndarray, shape (N,)   - temperature at each node
    """
    coords = np.array(mesh.points, dtype=np.float64)  # shape (N, 3)

    vel = np.array(mesh[EXODUS_VEL_NAME], dtype=np.float64)  # shape (N, 3)
    u = vel[:, 0]  # x-velocity
    v = vel[:, 1]  # y-velocity

    p = np.array(mesh[EXODUS_PRES_NAME], dtype=np.float64)  # shape (N,)
    T = np.array(mesh[EXODUS_TEMP_NAME], dtype=np.float64)  # shape (N,)

    return coords, u, v, p, T

def read_results(
        exodus_path: str | Path,
        geometry_config,
) -> dict[str, np.ndarray]:
    """
    Read a MOOSE Exodus II results file and map the fields onto the
    reference grid defined by GeometryConfig.

    Parameters
    ----------
    exodus_path : str | Path
        Path to the Exodus II file.
    geometry_config : GeometryConfig
        GeometryConfig object defining the reference grid.

    Returns
    -------
    dict with keys: 'occupancy', 'u', 'v', 'p', 'T'
        Each value is a (grid_ny, grid_nx) float32 ndarray.
        Solid cells hold self.solid_fill (NaN by default).
    """
    mesh = load_exodus_mesh(exodus_path)
    coords, u, v, p, T = extract_fields(mesh)

    gc = geometry_config
    occupancy = np.array(gc.occupancy_grid, dtype=np.float32)  # shape (grid_ny, grid_nx)
    binary = (occupancy >= gc.threshold).astype(np.float32)  # 1=solid, 0=fluid
    x_cen, y_cen = _build_grid_centers(gc)  # shape (grid_ny, grid_nx)

    node_xy = coords[:, :2] # drop z for 2D simulation
    node_fields = {"u": u, "v": v, "p": p, "T": T}

    field_grids = _map_to_grid(
        x_centers=x_cen,
        y_centers=y_cen,
        binary=binary,
        node_coords=node_xy,
        node_fields=node_fields,
    )
    field_grids["occupancy"] = binary
    return field_grids

def _build_grid_centers(gc) -> tuple[np.ndarray, np.ndarray]:
    """
    Build (grid_ny, grid_nx) arrays of cell-centre x and y
    coordinates, consistent with GeometryConfig.cell_center().
    
    x in [0, domain_length], y in [-domain_height/2, +domain_height/2].
    """
    dx = gc.domain_length / gc.grid_nx
    dy = gc.domain_height / gc.grid_ny

    # ix = 0..grid_nx-1, iy = 0..grid_ny-1
    ix = np.arange(gc.grid_nx)
    iy = np.arange(gc.grid_ny)

    x_1d = (ix + 0.5) * dx                          # shape (grid_nx,)
    y_1d = (iy + 0.5) * dy                          # shape (grid_ny,)

    x_centers = np.tile(x_1d[np.newaxis, :], (gc.grid_ny, 1))
    y_centers = np.tile(y_1d[:, np.newaxis], (1, gc.grid_nx))
    return x_centers, y_centers

def _map_to_grid(
        self,
        x_centers: np.ndarray,
        y_centers: np.ndarray,
        binary: np.ndarray,
        node_coords: np.ndarray,
        node_fields: dict[str, np.ndarray],
) -> dict[str, np.ndarray]:
    """
    Map unstructured node data onto the fixed reference grid.

    For each fluid cell: find the nearest FEM node (KD-tree) and
    copy its field values. Solid cells receive self.solid_fill (NaN by default).

    Parameters
    ----------
    x_centers : ndarray, shape (grid_ny, grid_nx)
        x-coordinates of cell centers.
    y_centers : ndarray, shape (grid_ny, grid_nx)
        y-coordinates of cell centers.
    binary : ndarray, shape (grid_ny, grid_nx)
        Binary occupancy grid (1=solid, 0=fluid).
    node_coords : ndarray, shape (N, 2)
        Unstructured node coordinates from the mesh.
    node_fields : dict
        Dictionary of unstructured field arrays at nodes.

    Returns
    -------
    dict with keys matching node_fields and values as (grid_ny, grid_nx) arrays.
    Solid cells are filled with self.solid_fill (NaN by default).
    """

    grid_ny, grid_nx = binary.shape

    grids = {
        name: np.full((grid_ny, grid_nx), self.solid_fill, dtype=np.float32)
        for name in node_fields
    }

    # Identify fluid cells
    fluid_iy, fluid_ix = np.where(binary < self.threshold)
    if fluid_iy.size == 0:
        return grids  # No fluid cells to map
    
    query_pts = np.column_stack([
        x_centers[fluid_iy, fluid_ix],
        y_centers[fluid_iy, fluid_ix]
    ]) # (n_fluid, 2)

    if self.interpolation == "nearest":
        self._map_nearest(grids, query_pts, fluid_iy, fluid_ix, node_coords, node_fields)
    elif self.interpolation == "linear":
        self._map_linear(grids, query_pts, fluid_iy, fluid_ix, node_coords, node_fields)
    else:
        raise ValueError(f"Unknown interpolation method: {self.interpolation}")
    
    return grids

def _map_nearest(self,
                    grids: dict,
                    query_pts: np.ndarray,
                    fluid_iy: np.ndarray,
                    fluid_ix: np.ndarray,
                    node_coords: np.ndarray,
                    node_fields: dict,
                    ) -> None:
    """
    KD-tree nearest-neighbor mapping of unstructured node data to grid cells. Modifies grids in-place.
    """
    tree = cKDTree(node_coords)
    distances, indices = tree.query(query_pts, k=1, workers=-1)

    # Sanity check: warn if any fluid cell is far from the nearest node, which may indicate a mapping issue.
    max_dist = distances.max()
    if max_dist > 0.5:
        print(f"[MOOSE Simulator] Warning: max distance from fluid cell to nearest node is {max_dist:.3f} m. Check mesh resolution.")

    for name, values in node_fields.items():
        grids[name][fluid_iy, fluid_ix] = values[indices].astype(np.float32)
    
def _map_linear(self,
                grids: dict,
                query_pts: np.ndarray,
                fluid_iy: np.ndarray,
                fluid_ix: np.ndarray,
                node_coords: np.ndarray,
                node_fields: dict,
                ) -> None:
    """
    Linear interpolation mapping of unstructured node data to grid cells using scipy.griddata. Modifies grids in-place.
    """
    for name, values in node_fields.items():
        interpolated = griddata(
            points=node_coords,
            values=values,
            xi=query_pts,
            method='linear',
            fill_value=self.solid_fill
        )
        nan_mask = np.isnan(interpolated)
        if nan_mask.any():
            # Fallback to nearest for points outside convex hull
            tree = cKDTree(node_coords)
            _, idx = tree.query(query_pts[nan_mask], k=1, workers=-1)
            interpolated[nan_mask] = values[idx]

        grids[name][fluid_iy, fluid_ix] = interpolated.astype(np.float32)